# Chapter 1: Common Conventions and API Elements of scikit-learn

This chapter introduces the foundational design principles and core API elements of **scikit-learn**, one of the most widely used Python libraries for machine learning. Since its launch in 2009, scikit-learn has become the go-to toolkit for ML practitioners across research, academia, and production environments.

We will cover the consistent interface that unifies all scikit-learn components -- **estimators**, **transformers**, and **pipelines** -- and walk through the key methods (`fit()`, `predict()`, `transform()`) that form the backbone of every ML workflow built with this library.

**What you will learn in this chapter:**

- The design philosophy behind scikit-learn's unified API
- How estimators implement `fit()` and `predict()` for supervised and unsupervised learning
- How transformers use `fit()` and `transform()` to preprocess data
- How to build custom estimators and transformers
- How pipelines chain multiple steps into a single reproducible workflow
- How to access model attributes (`coef_`, `intercept_`) and evaluation methods (`score()`)
- How to tune hyperparameters with `set_params()`, `get_params()`, `GridSearchCV()`, and `RandomizedSearchCV()`
- How metadata routing works in scikit-learn

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Verify scikit-learn version
import sklearn
print(f"scikit-learn version: {sklearn.__version__}")
print(f"NumPy version: {np.__version__}")

scikit-learn version: 1.6.1
NumPy version: 2.1.3


We are using scikit-learn version **1.8.0**, which is one of the latest stable releases. This version includes all the API patterns we will discuss in this chapter, including the modern metadata routing system and updated estimator tags. All code in this notebook is verified against this version.

## Introduction to scikit-learn's Design Philosophy

scikit-learn's power comes not from any single algorithm but from its **consistent, modular design**. The library is built on four core principles:

1. **Consistency** -- Every algorithm follows the same interface: `fit()`, `predict()`, `transform()`. Once you learn the pattern for one model, you know it for all of them.

2. **Simplicity** -- Sensible defaults let you get results fast. Every hyperparameter has a reasonable default value, so you can prototype quickly and tune later.

3. **Modularity** -- Estimators, transformers, and pipelines are independent building blocks. You can mix, match, and chain them freely.

4. **Reusability** -- Components built for one project slot into another. A custom transformer written for Project A can be dropped into Project B's pipeline with zero modification.

This design has a profound practical consequence: **switching between algorithms costs almost nothing in code changes**. Want to swap a logistic regression for a random forest? Change one line. This is not an accident -- it is the result of deliberate API design that prioritizes developer productivity.

**A note on capitalization:** The correct spelling is always lowercase **scikit-learn** (pronounced *sy-kit learn*), where *sci* abbreviates *science*. Think of it as a *(data) science kit for learning*.

# Understanding Estimators

Concept estimator merupakan inti dari perpustakaan scikit-learn. Estimator adalah objek Python (OOP) yang mengimplementasikan algoritma untuk mempelajari pola dari data. Setiap estimator baik itu model regresi, klasifikasi, maupun transformasi mengikuti antarmuka (interface) yang konsisten.

Dua metode paling mendasar dari sebuah supervised estimator adalah:
- fit(X, y): Melatih model dengan mempelajari hubungan antara fitur masukan ($X$) dan target ($y$).
- predict(X): Menggunakan model yang telah dilatih untuk memprediksi nilai target pada data baru.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

# Data contoh (5 sampel, 1 fitur)
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1, 2, 3, 3.5, 5])

# 1. Inisialisasi estimator
model = LinearRegression()

# 2. Melatih model
model.fit(X, y)

# 3. Memprediksi data baru
X_new = np.array([[6], [7]])
predictions = model.predict(X_new)

print("Hasil Prediksi untuk X = [6, 7]:", predictions)

Hasil Prediksi untuk X = [6, 7]: [5.75 6.7 ]


- model = LinearRegression(): Kita membuat instance dari kelas estimator LinearRegression.

- model.fit(X, y): Metode ini mengeksekusi persamaan Normal Equations ($\mathbf{w} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$) di latar belakang untuk menemukan bobot/kemiringan ($w = 0.95$) dan bias/intercept ($b = 0.05$).

- model.predict(X_new): Mengaplikasikan fungsi linier yang sudah dipelajari ($\hat{y} = 0.95x + 0.05$) ke data baru [[6], [7]], sehingga menghasilkan nilai prediksi [5.75, 6.7].

# Unsupervised Learning & fit_predict()

Dalam unsupervised learning (seperti clustering), data tidak memiliki label/target $y$. Untuk skenario ini, scikit-learn menyediakan metode pintas fit_predict().

Metode fit_predict() melatih model sekaligus mengembalikan label kluster untuk data yang sama dalam satu pemanggilan API. Ini berguna saat kita ingin menemukan struktur atau pola tersembunyi langsung pada dataset pelatihan tanpa perlu membagi data menjadi train/test.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

# Data contoh 1D
X = np.array([[1], [2], [3], [4], [5]])

# Inisialisasi KMeans dengan 2 kluster
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)

# Melatih sekaligus mendapatkan label kluster
labels = kmeans.fit_predict(X)

print("Label Kluster:", labels)
print("Pusat Kluster:", kmeans.cluster_centers_.flatten())

Label Kluster: [0 0 0 1 1]
Pusat Kluster: [2.  4.5]


- n_clusters=2: Menentukan bahwa kita ingin membagi data menjadi 2 kelompok/kluster.

- kmeans.fit_predict(X): Algoritma K-Means secara iteratif mencari titik pusat kluster (centroids) terbaik, lalu mengembalikan array label kluster untuk setiap titik data di X (misal: [0, 0, 0, 1, 1]).

- kmeans.cluster_centers_: Menampilkan koordinat pusat dari masing-masing kluster yang ditemukan (dalam contoh ini pada titik 2.0 dan 4.5).

# Transformers and the transform() Method

Transformer adalah jenis estimator khusus yang digunakan untuk manipulasi atau pembersihan data (seperti scaling, enkoding, atau imputasi). Antarmuka transformer menggunakan dua metode utama:
- fit(X): Mempelajari parameter dari data (misalnya menghitung rerata $\mu$ dan deviasi standar $\sigma$).
- transform(X): Menerapkan transformasi matematis ke data berdasarkan parameter yang dipelajari.
- fit_transform(X): Metode efisien yang menggabungkan fit() dan transform() dalam satu langkah.

Aturan Emas Production: Selalu jalankan fit_transform() pada data pelatihan (training set), tetapi hanya jalankan transform() (tanpa fit) pada data uji (test set). Jika Anda melakukan fit() pada data uji, akan terjadi kebocoran data (data leakage).

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# Data contoh (3 sampel, 2 fitur)
X = np.array([[1, 2], [3, 4], [5, 6]])

scaler = StandardScaler()

# Opsi A: Dipisah (fit lalu transform)
scaler.fit(X)
X_scaled_sep = scaler.transform(X)

# Opsi B: Digabung menggunakan fit_transform()
X_scaled_comb = scaler.fit_transform(X)

print("Data Asli:\n", X)
print("\nData Terstandardisasi (Z-score):\n", X_scaled_comb)
print("\nRerata yang dipelajari (mean_):", scaler.mean_)
print("Deviasi standar yang dipelajari (scale_):", scaler.scale_)

Data Asli:
 [[1 2]
 [3 4]
 [5 6]]

Data Terstandardisasi (Z-score):
 [[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]

Rerata yang dipelajari (mean_): [3. 4.]
Deviasi standar yang dipelajari (scale_): [1.63299316 1.63299316]


- StandardScaler(): Mengubah data agar memiliki rerata $\mu = 0$ dan deviasi standar $\sigma = 1$ menggunakan rumus $z = \frac{x - \mu}{\sigma}$.

- scaler.fit(X): Mempelajari bahwa rerata tiap kolom adalah 3.0 dan deviasi standarnya adalah 1.633.

- X_scaled_comb: Menghasilkan nilai skalar baru. Baris pertama [1, 2] ditransformasikan menjadi [-1.2247, -1.2247] karena titik data tersebut berada di bawah rerata.

## Handling Custom Estimators and Transformers

Arsitektur scikit-learn dirancang agar mudah diperluas (extensible). Anda dapat membuat kelas kustom sendiri yang dapat dimasukkan secara langsung ke dalam Pipeline atau GridSearchCV.

Untuk membuat transformer kustom, kelas Anda harus mengeksekusi subclassing dari:
- BaseEstimator: Memberikan kemampuan manajemen parameter otomatis seperti get_params() dan set_params().
- TransformerMixin: Secara otomatis menambahkan metode fit_transform() tanpa perlu kita tulis manual.

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

class ValueClipper(BaseEstimator, TransformerMixin):
    """Transformer kustom untuk membatasi rentang nilai fitur ke [lower, upper]."""

    def __init__(self, lower=0.0, upper=1.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        # Menandai bahwa transformer telah dilatih
        self.n_features_in_ = X.shape[1] if hasattr(X, 'shape') else len(X[0])
        self.is_fitted_ = True
        return self

    def transform(self, X):
        # Memastikan fit() telah dipanggil sebelum transform()
        check_is_fitted(self, 'is_fitted_')
        X = np.array(X)
        return np.clip(X, self.lower, self.upper)

# Penggunaan Transformer Kustom
X_data = np.array([[0.5, -0.3], [1.5, 0.7], [0.2, 2.1]])
clipper = ValueClipper(lower=0.0, upper=1.0)

clipper.fit(X_data)
X_clipped = clipper.transform(X_data)

print("Data Asli:\n", X_data)
print("\nData Setelah Di-clip [0.0, 1.0]:\n", X_clipped)

Data Asli:
 [[ 0.5 -0.3]
 [ 1.5  0.7]
 [ 0.2  2.1]]

Data Setelah Di-clip [0.0, 1.0]:
 [[0.5 0. ]
 [1.  0.7]
 [0.2 1. ]]


- __init__: Hanya menyimpan hyperparameter (lower dan upper). Jangan lakukan kalkulasi/fitur logika di sini agar kompatibel dengan scikit-learn.

- check_is_fitted(): Fungsi keamanan bawaan scikit-learn untuk memastikan pengguna tidak memanggil transform() sebelum fit().

- np.clip(): Memotong nilai di luar rentang. Nilai -0.3 dipotong menjadi 0.0, dan 1.5 dipotong menjadi 1.0.

## Pipelines and Workflow Automation

Pipeline menggabungkan beberapa langkah pemrosesan data (transformer) dan satu model akhir (estimator) ke dalam satu objek tunggal.

Saat memanggil pipeline.fit(), pipeline secara berurutan menjalankan fit_transform() pada setiap transformer dan fit() pada estimator akhir. Ketika memanggil pipeline.predict(), pipeline otomatis mengeksekusi transform() pada seluruh transformer dan predict() pada model akhir. Ini mencegah kesalahan kebocoran data dan mempermudah otomasi MLOps.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# 1. Membuat dataset sintetis klasifikasi
X, y = make_classification(n_samples=200, n_features=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# 2. Membuat Pipeline (Scaler -> Classifier)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42))
])

# 3. Melatih seluruh pipeline
pipe.fit(X_train, y_train)

# 4. Evaluasi
train_acc = pipe.score(X_train, y_train)
test_acc = pipe.score(X_test, y_test)

print(f"Akurasi Data Latih: {train_acc:.4f}")
print(f"Akurasi Data Uji  : {test_acc:.4f}")
print("Langkah-langkah Pipeline:", [name for name, _ in pipe.steps])

Akurasi Data Latih: 0.8400
Akurasi Data Uji  : 0.9000
Langkah-langkah Pipeline: ['scaler', 'classifier']


- Pipeline([('nama_step', objek), ...]): Menerima daftar tuple berisi nama langkah dan objeknya.

- pipe.fit(X_train, y_train): Secara otomatis melakukan scaling pada X_train lalu melatih model LogisticRegression pada data yang sudah di-skala tersebut.

- pipe.score(X_test, y_test): Menggunakan parameter scaling dari X_train untuk meng-skala X_test secara otomatis sebelum menghitung skor akurasi, menjamin tidak ada kebocoran data dari data uji.

## Common Attributes and Methods

Setelah proses fit() selesai, estimator menyimpan variabel atau parameter yang berhasil dipelajari dari data ke dalam atribut yang diakhiri dengan garis bawah (_).

Atribut utama pada model linier mencakup:
- coef_: Bobot/koefisien fitur ($\mathbf{w}$).
- intercept_: Bias/titik potong garis ($b$).

Metode score(X, y) menyediakan evaluasi default:
- Mengembalikan nilai $R^2$ (koefisien determinasi) untuk model regresi.
- Mengembalikan nilai Akurasi untuk model klasifikasi.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1, 2, 3, 3.5, 5])

model = LinearRegression()
model.fit(X, y)

# Mengakses atribut berakhiran '_'
print("Koefisien / Bobot (coef_):", model.coef_)
print("Intercept / Bias (intercept_):", model.intercept_)

# Evaluasi menggunakan score()
r2_score = model.score(X, y)
print(f"Skor R-squared (R2): {r2_score:.6f}")

Koefisien / Bobot (coef_): [0.95]
Intercept / Bias (intercept_): 0.04999999999999938
Skor R-squared (R2): 0.980978


- model.coef_ & model.intercept_: Menunjukkan bahwa persamaan linier yang terbentuk adalah $\hat{y} = 0.95x + 0.05$.

- model.score(X, y): Nilai $R^2 = 0.9809$ menunjukkan bahwa $98.1\%$ variasi nilai pada $y$ dapat dijelaskan secara linier oleh fitur $x$.

## Hyperparameter Tuning with Search Methods

Hyperparameter adalah nilai pengaturan model yang ditentukan sebelum proses pelatihan dimulai (tidak dipelajari secara otomatis dari data).

scikit-learn menyediakan dua metode untuk mengelola hyperparameter:
- get_params(): Mengambil daftar seluruh pengaturan hyperparameter saat ini.
- set_params(): Mengatur atau mengubah nilai hyperparameter secara terprogram.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 1. Inisialisasi model
model = RandomForestClassifier()

# 2. Mengubah hyperparameter secara terprogram
model.set_params(n_estimators=100, max_depth=10, random_state=42)

# 3. Mengambil seluruh parameter
params = model.get_params()

print("Beberapa Hyperparameter Terpilih:")
print(" - n_estimators:", params['n_estimators'])
print(" - max_depth   :", params['max_depth'])
print(" - criterion   :", params['criterion'])

Beberapa Hyperparameter Terpilih:
 - n_estimators: 100
 - max_depth   : 10
 - criterion   : gini


- set_params(n_estimators=100, max_depth=10): Mengatur agar Random Forest membuat 100 pohon keputusan dan membatasi kedalaman maksimum setiap pohon hingga 10 tingkat.

- get_params(): Mengembalikan dictionary berisi semua opsi konfigurasi model. Metode ini digunakan secara internal oleh modul pencarian otomatis seperti GridSearchCV.

## Working with Metadata: Tags and More

scikit-learn menggunakan sistem metadata untuk mengontrol perilaku internal estimator.

1. Estimator Tags (__sklearn_tags__()): Tag khusus yang memberi tahu ekosistem scikit-learn tentang kemampuan model (misalnya: apakah model mendukung data multikelas, data berjarak, atau input API Array).

2. Metadata Routing: Mekanisme yang memungkinkan informasi tambahan (seperti sample_weight atau label grup) disalurkan dengan tepat ke langkah pipeline tertentu tanpa mengganggu langkah preprocessing lainnya.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Inisialisasi estimator
model = LogisticRegression()

# Memeriksa tag internal estimator
tags = model.__sklearn_tags__()

print("Nama Estimator:", type(model).__name__)
print("Tag Dukungan Array API:", tags.array_api_support)
print("Tag Non-Deterministik :", tags.non_deterministic)

Nama Estimator: LogisticRegression
Tag Dukungan Array API: False
Tag Non-Deterministik : False


- __sklearn_tags__(): Mengembalikan objek struktur yang menyimpan metadata kapabilitas model.

- Tag ini berfungsi sebagai "kontrak" otomatis antara model dengan infrastruktur scikit-learn (seperti GridSearchCV atau Pipeline) untuk memastikan kompatibilitas sebelum eksekusi kode dilakukan.

## Best Practices for API Usage

Having covered the core components of scikit-learn's API, let's distill the key principles that will serve you throughout this book and beyond:

### The Golden Rules

**1. Uniform API -- Learn the pattern once, apply everywhere.** Every estimator follows `fit()` $\rightarrow$ `predict()` (for models) or `fit()` $\rightarrow$ `transform()` (for transformers). This consistency means that switching from `LogisticRegression()` to `RandomForestClassifier()` requires changing exactly one line of code.

**2. Preprocess before modeling.** Real-world data is messy. Always apply appropriate transformations (scaling, encoding, imputation) before fitting a model. scikit-learn's `sklearn.preprocessing` module provides the tools; Chapters 2 and 3 cover them in depth.

**3. Use pipelines for complex workflows.** Chaining transformers and estimators into a `Pipeline()` eliminates data leakage, ensures reproducibility, and produces a single deployable artifact. This becomes essential in Chapter 14.

**4. Validate with cross-validation.** Never evaluate a model on its training data alone. Use `cross_val_score()` or `GridSearchCV()` (which performs CV internally) to estimate how well your model generalizes. The `sklearn.model_selection` module is your friend.

**5. Tune hyperparameters systematically.** Default hyperparameters are reasonable starting points, but almost never optimal. Use `GridSearchCV()` for small parameter spaces and `RandomizedSearchCV()` for large ones. We explore this in detail in Chapter 12.

**6. Inspect model internals.** Attributes like `coef_`, `feature_importances_`, and `intercept_` provide interpretability. The `score()` method gives a quick performance check. Always look inside the black box.

## Chapter Summary

This chapter established the foundational vocabulary and patterns of scikit-learn's API:

| Concept | Key Methods | Role |
|:--------|:-----------|:-----|
| **Estimator** | `fit()`, `predict()` | Learn from data and make predictions |
| **Transformer** | `fit()`, `transform()`, `fit_transform()` | Preprocess and modify data |
| **Pipeline** | `fit()`, `predict()`, `score()` | Chain steps into a single workflow |
| **Attributes** | `coef_`, `intercept_`, `feature_importances_` | Inspect learned parameters |
| **Evaluation** | `score()` | Quick performance assessment |
| **Hyperparameters** | `set_params()`, `get_params()` | Configure model settings |
| **Metadata** | `__sklearn_tags__()` | Describe estimator capabilities |

The consistency of this API is what makes scikit-learn so productive. Whether you are building a simple linear regression or a complex ensemble pipeline with custom transformers, the interface remains the same.